# 02 — Add Unity Catalog Metadata

Adds primary key constraints, table comments, and column comments to all 3 tables for data governance, discoverability, and Genie context.

In [0]:
dbutils.widgets.text("catalog", "medtech", "Catalog")
dbutils.widgets.text("schema",  "sales",   "Schema")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

print("catalog:", catalog)
print("schema:",  schema)

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);
USE SCHEMA IDENTIFIER(:schema);

## Primary Keys

In [0]:
spark.sql(f"ALTER TABLE {catalog}.{schema}.hcp_procedure_volume ALTER COLUMN npi SET NOT NULL")
spark.sql(f"ALTER TABLE {catalog}.{schema}.hcp_procedure_volume ADD CONSTRAINT pk_hcp_procedure_volume PRIMARY KEY (npi)")
spark.sql(f"ALTER TABLE {catalog}.{schema}.product_upgrades ALTER COLUMN row_number SET NOT NULL")
spark.sql(f"ALTER TABLE {catalog}.{schema}.product_upgrades ADD CONSTRAINT pk_product_upgrades PRIMARY KEY (row_number)")
spark.sql(f"ALTER TABLE {catalog}.{schema}.account_targeting ALTER COLUMN row_number SET NOT NULL")
spark.sql(f"ALTER TABLE {catalog}.{schema}.account_targeting ADD CONSTRAINT pk_account_targeting PRIMARY KEY (row_number)")
print("Primary keys added")

## Table Comments

In [0]:
spark.sql(f"COMMENT ON TABLE {catalog}.{schema}.hcp_procedure_volume IS 'Healthcare provider (HCP/surgeon) procedure volumes by NPI. Contains current year (CY) and prior year (PY) procedure volumes and max market opportunity by product line (Alpha, Beta, Gamma, Delta). Join to other tables via rep_id and account.'")
spark.sql(f"COMMENT ON TABLE {catalog}.{schema}.product_upgrades IS 'Product upgrade opportunities by account, territory, and product. Contains upgrade codes, opportunity dollar amounts, discontinuation dates, and rolling 12-month sales. Product lines: Alpha, Beta, Gamma, Delta. Categories: Accessories, Capital Equipment, Service Contract, Disposables.'")
spark.sql(f"COMMENT ON TABLE {catalog}.{schema}.account_targeting IS 'Account-level strategic targeting with penetration metrics. Contains target types (Tier 1=Upgrade, Tier 2=Competitive, Tier 3=Market Expansion), GPO affiliations, clinical focus areas, and year-over-year penetration trends. Key for identifying high-value opportunities.'")
print("Table comments added")

## Column Comments — account_targeting

In [0]:
comments = {
    "row_number":               "Unique row identifier (primary key)",
    "area":                     "Geographic area (e.g., Northeast, Southeast, Central, Northwest, Southwest)",
    "region":                   "Geographic region within an area",
    "territory":                "Sales territory assigned to a rep",
    "rep_id":                   "Sales representative identifier - shared across all 3 tables",
    "idn":                      "Integrated Delivery Network - hospital system name",
    "account":                  "Hospital or facility account name - shared across all 3 tables",
    "clinical_focus_area":      "Clinical specialty or focus area for targeting",
    "target_type":              "Sales strategy tier: Tier 1 = Upgrade, Tier 2 = Competitive, Tier 3 = Market Expansion, Non-Target = not targeted",
    "product_line":             "Product line: Alpha Series, Beta Series, Gamma Series, or Delta Series",
    "category":                 "Product category within a product line",
    "gpo":                      "Group Purchasing Organization affiliation",
    "total_units_sold":         "Total number of units sold to this account",
    "opportunity":              "Dollar opportunity value ($) for this account/product/focus combination",
    "net_cost_to_customer":     "Net cost to customer in dollars ($) after discounts",
    "rolling_12_sales":         "Rolling 12-month sales in dollars ($)",
    "num_codes_on_account_shelf": "Number of product codes on the account shelf",
    "num_codes_in_system":      "Number of product codes in the hospital system",
    "num_codes_in_neither":     "Number of product codes not on shelf or in system",
    "penetration_2025":         "Product penetration percentage for 2025 (decimal 0.0 to 1.0, multiply by 100 for %)",
    "penetration_2024":         "Product penetration percentage for 2024 (decimal 0.0 to 1.0)",
    "penetration_2023":         "Product penetration percentage for 2023 (decimal 0.0 to 1.0)",
    "trend_3_month":            "3-month sales trend indicator",
    "trend_6_month":            "6-month sales trend indicator",
    "trend_9_month":            "9-month sales trend indicator",
    "cy_market_exposure":       "Current year market exposure percentage",
    "py_market_exposure":       "Prior year market exposure percentage",
}
for col, comment in comments.items():
    spark.sql(f"ALTER TABLE {catalog}.{schema}.account_targeting ALTER COLUMN {col} COMMENT '{comment}'")
print("account_targeting column comments added")

## Column Comments — product_upgrades

In [0]:
comments = {
    "row_number":               "Unique row identifier (primary key)",
    "area":                     "Geographic area (e.g., Northeast, Southeast, Central, Northwest, Southwest)",
    "region":                   "Geographic region within an area",
    "territory":                "Sales territory assigned to a rep",
    "idn":                      "Integrated Delivery Network - hospital system name",
    "rep_id":                   "Sales representative identifier - shared across all 3 tables",
    "account":                  "Hospital or facility account name - shared across all 3 tables",
    "product_line":             "Product line: Alpha Series, Beta Series, Gamma Series, or Delta Series",
    "product_category":         "Product category (e.g., Accessories, Capital Equipment, Service Contract, Disposables)",
    "product_description":      "Specific product name/description within a category",
    "comped_status":            "Whether the product is provided at no cost: Comped or Not Comped",
    "segment":                  "Market segment classification",
    "num_upgrade_codes":        "Number of upgrade product codes available",
    "num_codes_on_account_shelf": "Number of product codes on the account shelf",
    "num_codes_in_idn_system":  "Number of product codes in the IDN hospital system",
    "num_codes_in_neither":     "Number of product codes not on shelf or in system",
    "discontinuation_date":     "Date when the product will be or was discontinued",
    "cy_opportunity":           "Current year dollar opportunity value ($) for this product upgrade",
    "py_opportunity":           "Prior year dollar opportunity value ($) for this product upgrade",
    "net_cost_to_customer":     "Net cost to customer in dollars ($) after discounts",
    "rolling_12_sales":         "Rolling 12-month sales in dollars ($)",
}
for col, comment in comments.items():
    spark.sql(f"ALTER TABLE {catalog}.{schema}.product_upgrades ALTER COLUMN {col} COMMENT '{comment}'")
print("product_upgrades column comments added")

## Column Comments — hcp_procedure_volume

In [0]:
comments = {
    "npi":                      "National Provider Identifier - unique ID for each healthcare provider (primary key)",
    "surgeon_name":             "Full name of the surgeon/HCP",
    "rep_id":                   "Sales representative identifier - shared across all 3 tables",
    "account":                  "Hospital or facility account name - shared across all 3 tables",
    "specialty":                "Medical specialty of the surgeon (e.g., General Surgery, Orthopedics)",
    "area":                     "Geographic area (e.g., Northeast, Southeast, Central, Northwest, Southwest)",
    "cy_procedure_volume":      "Current year total procedure volume count",
    "py_procedure_volume":      "Prior year total procedure volume count",
    "cy_vs_py_procedure_volume": "Year-over-year change in procedure volume (CY minus PY)",
    "cy_alpha_max_market":      "Current year max market opportunity ($) for Alpha Series products",
    "cy_beta_max_market":       "Current year max market opportunity ($) for Beta Series products",
    "cy_gamma_max_market":      "Current year max market opportunity ($) for Gamma Series products",
    "cy_delta_max_market":      "Current year max market opportunity ($) for Delta Series products",
    "total_cy_max_market":      "Total current year max market opportunity ($) across all product lines",
    "py_alpha_max_market":      "Prior year max market opportunity ($) for Alpha Series products",
    "py_beta_max_market":       "Prior year max market opportunity ($) for Beta Series products",
    "py_gamma_max_market":      "Prior year max market opportunity ($) for Gamma Series products",
    "py_delta_max_market":      "Prior year max market opportunity ($) for Delta Series products",
    "total_py_max_market":      "Total prior year max market opportunity ($) across all product lines",
}
for col, comment in comments.items():
    spark.sql(f"ALTER TABLE {catalog}.{schema}.hcp_procedure_volume ALTER COLUMN {col} COMMENT '{comment}'")
print("hcp_procedure_volume column comments added")

## Verify Metadata

In [0]:
%sql
DESCRIBE EXTENDED hcp_procedure_volume;